# Generate the files

## Requirements

[rustup](https://rustup.rs/)
[python3](https://www.python.org/downloads/)

Maybe virtual env?

## Step by Step
We need to locate the directory where the replays are, in my case I dual-boot and mount windows to `/mnt/windows`

The final step takes around 32 seconds to process 3600 Replay files. This means about 110 replays per-second on a 16 Core machine.
```bash
$ git clone https://github.com/sebosp/s2protocol-rs
$ cd s2protocol-rs
$ mkdir ipcs/
$ cargo run --features arrow,syntax -r -- -v error --timing --source /home/seb/SCReplaysOnNVMe --output /home/seb/git/s2protocol-rs/ipcs/ write-arrow-ipc --process-max-files 10000000
36752 files have valid init data, processing...
Total time: 234.08368818s
$ du -sh ipcs/*
1.8G    ipcs/cmd_target_point.ipc
750M    ipcs/cmd_target_unit.ipc
9.7M    ipcs/details.ipc
112M    ipcs/lobby_init_data.ipc
2.1G    ipcs/stats.ipc
9.3G    ipcs/unit_born.ipc
5.5G    ipcs/unit_died.ipc
172M    ipcs/upgrades.ipc
36M     ipcs/user_init_data.ipc
$ cd ../s2-polars-data-analysis/
$ pip install -r requirements.txt
$ jupyter lab --notebook-dir=jupyter_notebooks/
# Open the URL explained in the terminal, this will open this notebook for interacting with the data.
```


In [1]:
# Updated to the s2protocol-rs 3.6.0 snapshot, partitioned by day and keyed by ext_fs_id.
# import datashader as ds
import plotly.express as px
import polars as pl

# Reads the snapshot, see its docstrings.
import snapshot as s2

pl.Config.set_tbl_width_chars(256)
pl.Config.set_fmt_str_lengths(256)
pl.Config.set_tbl_rows(24)

# The snapshot, `$S2_IPC_DIR` or `ipcs` in the replay directory when written by swarmy.
ipc_dir = s2.ipc_dir()

# Filter out these:
# Beacon*: Similiar to above, default targets of hatcheries and buildings
#          points to Beacons, clicks on the map, or actions when teams play
#          together and send attack/defend points on the map.
#          For now we'll avoid them.

unit_born_df = s2.scan(ipc_dir, "unit_born")
# .filter((~pl.col("unit_type_name").str.starts_with("Beacon")))
unit_died_df = s2.scan(ipc_dir, "unit_died")
# .filter((~pl.col("unit_died_name").str.starts_with("Beacon")))
stats_df = s2.scan(ipc_dir, "stats")
upgrades_df = s2.scan(ipc_dir, "upgrades")
details_df = s2.scan(ipc_dir, "details")
cmd_target_point_df = s2.scan(ipc_dir, "cmd_target_point")
cmd_target_unit_df = s2.scan(ipc_dir, "cmd_target_unit")
# The lobby slots and the user init data are no longer separate files, the lobby slot of each
# player is in the lobby_slot_* columns of the details.
lobby_slot_df = details_df.select(
    "ext_fs_id", "ext_fs_file_name", "player_name", "player_working_set_slot_id", pl.col("^lobby_slot_.*$")
)

# Find a game:
lobby_slot_df.filter(
    pl.col("ext_fs_file_name").str.contains("Grand Finals")
    & pl.col("ext_fs_file_name").str.contains("2025")
    & pl.col("lobby_slot_observe").eq(0)
).collect()

ext_fs_id,ext_fs_file_name,player_name,player_working_set_slot_id,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap
u64,str,str,u8,i64,u8,u8,u8,i64,i64,u32
34302,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game1 - Classic vs Serral - Persephone.SC2Replay""","""Serral""",2,2,0,160,184,3,0,100
34302,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game1 - Classic vs Serral - Persephone.SC2Replay""","""Classic""",12,3,0,160,184,3,0,100
34303,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game2 - Classic vs Serral - Incorporeal.SC2Replay""","""Serral""",11,2,0,152,176,3,0,100
34303,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game2 - Classic vs Serral - Incorporeal.SC2Replay""","""Classic""",12,3,0,152,176,3,0,100
34304,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game3 - Classic vs Serral - Ultralove.SC2Replay""","""Classic""",11,2,0,184,184,3,0,100
34304,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game3 - Classic vs Serral - Ultralove.SC2Replay""","""Serral""",12,3,0,184,184,3,0,100
34305,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game4 - Classic vs Serral - Pylon.SC2Replay""","""Serral""",11,2,0,248,248,3,0,100
34305,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game4 - Classic vs Serral - Pylon.SC2Replay""","""Classic""",14,5,0,248,248,3,0,100
34306,"""/home/seb/SCReplaysOnNVMe/2025EWCReplayPack/3 - Playoffs/8 - Grand Finals - Serral vs Classic/20250725 - Game5 - Classic vs Serral - Ley Lines.SC2Replay""","""Serral""",11,2,0,200,184,3,0,100


In [2]:
chosen_fs_id = 16594  # One of the above

In [3]:
lobby_slot_df.collect_schema()

Schema([('ext_fs_id', UInt64),
        ('ext_fs_file_name', String),
        ('player_name', String),
        ('player_working_set_slot_id', UInt8),
        ('lobby_slot_user_id', Int64),
        ('lobby_slot_observe', UInt8),
        ('lobby_slot_map_size_x', UInt8),
        ('lobby_slot_map_size_y', UInt8),
        ('lobby_slot_difficulty', Int64),
        ('lobby_slot_ai_build', Int64),
        ('lobby_slot_handicap', UInt32)])

In [4]:
lobby_slot_df.describe()

statistic,ext_fs_id,ext_fs_file_name,player_name,player_working_set_slot_id,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap
str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",91547.0,"""91547""","""91547""",90646.0,76027.0,91547.0,91547.0,91547.0,91547.0,91547.0,91547.0
"""null_count""",0.0,"""0""","""0""",901.0,15520.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",18120.033983,null,null,3.257452,1.718153,0.0,186.885425,186.2101,3.416671,0.251062,99.953903
"""std""",10434.438767,null,null,4.537205,1.992008,0.0,42.214121,43.092774,1.892292,2.353903,1.480307
"""min""",0.0,"""/home/seb/SCReplaysOnNVMe/2023_07-ESL_SC2_Masters_Winter_2023_Finals/1 - Winners Stage/A1 - UB Ro8 #1 - GuMiho vs ShoWTimE/20230616 - Game 2 - ShoWTimE vs GuMiho - PvT - Hecate.SC2Replay""","""&lt;0001&gt;<sp/>VOODOO""",0.0,0.0,0.0,0.0,0.0,1.0,0.0,50.0
"""25%""",8870.0,null,null,0.0,0.0,0.0,176.0,176.0,3.0,0.0,100.0
"""50%""",18192.0,null,null,1.0,1.0,0.0,192.0,192.0,3.0,0.0,100.0
"""75%""",26931.0,null,null,3.0,3.0,0.0,200.0,208.0,3.0,0.0,100.0
"""max""",36393.0,"""/home/seb/SCReplaysOnNVMe/wormgat/Multiplayer/Fields of Death CE.SC2Replay""","""호에엥""",15.0,15.0,0.0,248.0,248.0,10.0,132.0,100.0


In [5]:
lobby_slot_df.select(pl.col("lobby_slot_user_id").value_counts(sort=True)).collect()
# - observe 0, 1, 2 --- 0 = player
# - control 0,1,2,3 --- unsure, most are 1.
# - user_id null, 0, 1..15 --- unsure, most are null (155Ks), then 0, 12Ks = matches number of files.
# - team_id 0, 1 --- seems for players
# - working_set_slot_id 0..15 --- unsure (are these maybe the slots of 16 players that can be in a lobby?
#   For example active players can be 12, but doesn't match the event player ids
# - map_size_x: 0? 1..16 possible values of size_x
# - map_size_y: 0? 1.. many more than 16 possible values? doesn't match size_x hmm18780

lobby_slot_user_id
struct[2]
"{0,24080}"
"{1,22797}"
"{null,15520}"
"{2,8951}"
"{3,8317}"
"{4,4592}"
"{5,2859}"
"{6,2002}"
"{7,945}"


In [6]:
lobby_slot_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id) & pl.col("lobby_slot_observe").eq(0)).collect()

ext_fs_id,ext_fs_file_name,player_name,player_working_set_slot_id,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap
u64,str,str,u8,i64,u8,u8,u8,i64,i64,u32
16594,"""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""","""Hallo""",0,0,0,232,232,1,0,100
16594,"""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""","""Sazed""",1,1,0,232,232,1,0,100
16594,"""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""","""A.I. 2 (Elite)""",2,null,0,232,232,7,0,100
16594,"""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""","""A.I. 3 (Elite)""",3,null,0,232,232,7,0,100


In [7]:
details_df.collect_schema()

Schema([('player_name', String),
        ('player_toon_region', UInt8),
        ('player_toon_program_id', UInt32),
        ('player_toon_realm', UInt32),
        ('player_toon_id', UInt64),
        ('player_race', String),
        ('player_color_a', UInt8),
        ('player_color_r', UInt8),
        ('player_color_g', UInt8),
        ('player_color_b', UInt8),
        ('player_control', UInt8),
        ('player_team_id', UInt8),
        ('player_observe', UInt8),
        ('player_result', String),
        ('player_working_set_slot_id', UInt8),
        ('player_hero', String),
        ('title', String),
        ('map_info_sha256', String),
        ('is_blizzard_map', Boolean),
        ('time_utc', Int64),
        ('time_local_offset', Int64),
        ('lobby_slot_user_id', Int64),
        ('lobby_slot_observe', UInt8),
        ('lobby_slot_map_size_x', UInt8),
        ('lobby_slot_map_size_y', UInt8),
        ('lobby_slot_difficulty', Int64),
        ('lobby_slot_ai_build', Int64),
   

In [8]:
details_df.describe()

statistic,player_name,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_observe,player_result,player_working_set_slot_id,player_hero,title,map_info_sha256,is_blizzard_map,time_utc,time_local_offset,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap,player_scaled_rating,lobby_random_seed,cache_handle_region,cache_handle_extension,cache_handles,ext_fs_sha256,ext_fs_file_name,ext_fs_id,ext_datetime,ext_protocol_version
str,str,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,f64,str,f64
"""count""","""91547""",91547.0,91547.0,91547.0,91547.0,"""91547""",91547.0,91547.0,91547.0,91547.0,91547.0,91547.0,91547.0,"""91547""",90646.0,"""91547""","""91547""","""91547""",91547.0,91547.0,91547.0,76027.0,91547.0,91547.0,91547.0,91547.0,91547.0,91547.0,36006.0,91547.0,"""91547""","""91547""","""91547""","""91547""","""91547""",91547.0,"""91547""",91547.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""",901.0,"""0""","""0""","""0""",0.0,0.0,0.0,15520.0,0.0,0.0,0.0,0.0,0.0,0.0,55541.0,0.0,"""0""","""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,1.506756,17620.804811,0.875277,5.3189e6,null,254.9509,89.774619,58.663168,142.891608,2.16953,0.512524,0.0,null,3.257452,null,null,null,0.818847,1.3309e17,3.5989e10,1.718153,0.0,186.885425,186.2101,3.416671,0.251062,99.953903,-5651.47081,2.1433e9,null,null,null,null,null,18120.033983,"""2022-10-02 16:26:23.114821""",87817.605798
"""std""",null,1.430902,8049.586727,0.452805,4.5750e6,null,1.232769,89.262005,53.334531,108.045824,0.375222,0.584674,0.0,null,4.537205,null,null,null,null,5.0596e14,1.4774e11,1.992008,0.0,42.214121,43.092774,1.892292,2.353903,1.480307,16687.491517,1.2421e9,null,null,null,null,null,10434.438767,null,4256.990871
"""min""","""&lt;0001&gt;<sp/>VOODOO""",0.0,0.0,0.0,0.0,"""""",224.0,0.0,0.0,0.0,2.0,0.0,0.0,"""Loss""",0.0,"""""","""$$$Fastest Map$$$""","""""",0.0,1.3210e17,-3.6000e11,0.0,0.0,0.0,0.0,1.0,0.0,50.0,-36560.0,42748.0,"""""","""""","""""","""00002a9dc36528452ad1938d035d16186b31390e6faef0487a0468214d644b34""","""/home/seb/SCReplaysOnNVMe/2023_07-ESL_SC2_Masters_Winter_2023_Finals/1 - Winners Stage/A1 - UB Ro8 #1 - GuMiho vs ShoWTimE/20230616 - Game 2 - ShoWTimE vs GuMiho - PvT - Hecate.SC2Replay""",0.0,"""2019-08-13 22:39:44.775573""",75689.0
"""25%""",null,1.0,21298.0,1.0,1.156381e6,null,255.0,0.0,20.0,30.0,2.0,0.0,0.0,null,0.0,null,null,null,null,1.3266e17,3.6000e10,0.0,0.0,176.0,176.0,3.0,0.0,100.0,2025.0,1.0652e9,null,null,null,null,null,8870.0,"""2021-05-21 14:34:26.932465""",84643.0
"""50%""",null,2.0,21298.0,1.0,4.165877e6,null,255.0,51.0,66.0,129.0,2.0,0.0,0.0,null,1.0,null,null,null,null,1.3301e17,3.6000e10,1.0,0.0,192.0,192.0,3.0,0.0,100.0,3085.0,2.1323e9,null,null,null,null,null,18192.0,"""2022-07-02 07:54:04.872084""",87702.0
"""75%""",null,2.0,21298.0,1.0,8.854336e6,null,255.0,180.0,66.0,255.0,2.0,1.0,0.0,null,3.0,null,null,null,null,1.3346e17,7.2000e10,3.0,0.0,200.0,208.0,3.0,0.0,100.0,3729.0,3.2143e9,null,null,null,null,null,26931.0,"""2023-12-01 15:15:12.466319""",91115.0
"""max""","""호에엥""",98.0,21298.0,2.0,2.0889428e7,"""프로토스""",255.0,255.0,255.0,255.0,3.0,14.0,0.0,"""Win""",15.0,"""""","""해비테이션 스테이션 - 래더""","""ffaecc032d9ca0e2c02b218b1bf80bccad10ba229e7edab734f29c7575fa0995""",1.0,1.3437e17,4.6800e11,15.0,0.0,248.0,248.0,10.0,132.0,100.0,7416.0,4.2947e9,"""XX""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,eaceeb172ee73b9650789d2fee249bac54ecee8f2b2204980929d69aeb135a44,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505…","""fffe9e9afdd6676125f0204935343add1386f924147125e050bdea4838f60d9f""

In [9]:
details_df.filter(pl.col("player_toon_id") != 0).select(
    pl.col("player_name").value_counts(sort=True)
).collect()

# player_name contains the clan, clan also in user_inuser_init_data_df separate from name, but no user_id/player_id there. Can be cleaned with pl.col("player_name").str.split(by="<sp/>").list.last().alias("player_name")
# player_toon_id may be 0 for A.I. game practice.
# player_race is Protoss, Terran, but also has hanzi (神族), korean? (테란) and strings like "Primal Zerg" or "Terraner" or ""
# player_team_id is either 0 or 1
# player_observe is always 0
# player_result can be 1, 2,0 and 3
#               1 and 2 are win,loss (not sure which is which).
#               0 is probably undecided (or maybe teh status of observer?)
#               3 is probably tie (there are only 50 games with this status in this 12K replay dataset.
# player_working_set_slot_id null, 0..15
#                            not the same as lobby_slot_init_data_df working_set_slot_id
# player_hero always ""
# title is the name of the map
# is_blizzard_map seems filled
# time_utc is that weird "epoch" after 2000 or something
# time_local_offset is +- in millis (or nanos?)

player_name
struct[2]
"{""&lt;Mealen&gt;<sp/>Doombringer"",3259}"
"{""&lt;chezs&gt;<sp/>Sazed"",3019}"
"{""Sazed"",2181}"
"{""Massa"",1661}"
"{""qqq"",1374}"
"{""&lt;ƖIıIƖ&gt;<sp/>Sazed"",744}"
"{""ShoWTimE"",594}"
"{""Serral"",574}"
"{""Reynor"",573}"


In [10]:
# Remove the clan from the player_name
# the clan and player are separated already in the user init data
# but I don't see a way to tie it to the lobby_state user data).
details_df = details_df.with_columns(
    pl.col("player_name").str.split(by="<sp/>").list.last().alias("player_name")
)
details_df.filter(pl.col("player_toon_id") != 0).select(
    pl.col("player_name").value_counts(sort=True)
).collect()

player_name
struct[2]
"{""Sazed"",5969}"
"{""Doombringer"",3259}"
"{""Massa"",1661}"
"{""qqq"",1374}"
"{""Reynor"",1006}"
"{""Serral"",938}"
"{""ShoWTimE"",750}"
"{""HeroMarine"",737}"
"{""ByuN"",704}"


In [11]:
# Each row of the details has the lobby slot of the player already. The tracker events (upgrades,
# stats, unit_born, unit_died) identify the player by player_id, mapped from unit_born by
# s2.players(). The game events (cmd_target_*) by user_id, the lobby slot user id.
details_lobby_df = (
    details_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id))
    .join(
        s2.players(ipc_dir).select("ext_fs_id", "player_name", "player_id"),
        on=["ext_fs_id", "player_name"],
        how="left",
    )
    .with_columns(pl.col("lobby_slot_user_id").alias("user_id"))
)
details_lobby_df.collect_schema()

Schema([('player_name', String),
        ('player_toon_region', UInt8),
        ('player_toon_program_id', UInt32),
        ('player_toon_realm', UInt32),
        ('player_toon_id', UInt64),
        ('player_race', String),
        ('player_color_a', UInt8),
        ('player_color_r', UInt8),
        ('player_color_g', UInt8),
        ('player_color_b', UInt8),
        ('player_control', UInt8),
        ('player_team_id', UInt8),
        ('player_observe', UInt8),
        ('player_result', String),
        ('player_working_set_slot_id', UInt8),
        ('player_hero', String),
        ('title', String),
        ('map_info_sha256', String),
        ('is_blizzard_map', Boolean),
        ('time_utc', Int64),
        ('time_local_offset', Int64),
        ('lobby_slot_user_id', Int64),
        ('lobby_slot_observe', UInt8),
        ('lobby_slot_map_size_x', UInt8),
        ('lobby_slot_map_size_y', UInt8),
        ('lobby_slot_difficulty', Int64),
        ('lobby_slot_ai_build', Int64),
   

In [12]:
upgrades_df.collect_schema()

Schema([('player_id', UInt8),
        ('name', String),
        ('count', Int32),
        ('ext_replay_loop', Int64),
        ('ext_replay_seconds', UInt32),
        ('ext_fs_id', UInt64)])

In [13]:
upgrades_df.describe()

statistic,player_id,name,count,ext_replay_loop,ext_replay_seconds,ext_fs_id
str,f64,str,f64,f64,f64,f64
"""count""",3.128158e6,"""3128158""",3.128158e6,3.128158e6,3.128158e6,3.128158e6
"""null_count""",0.0,"""0""",0.0,0.0,0.0,0.0
"""mean""",3.117687,null,0.68734,9047.620321,403.58311,19542.821498
"""std""",3.036174,null,1.395386,13843.097019,617.890548,9275.784711
"""min""",0.0,"""05EnergyRegen""",-64.0,0.0,0.0,0.0
"""25%""",1.0,null,0.0,0.0,0.0,11967.0
"""50%""",2.0,null,1.0,5127.0,228.0,22524.0
"""75%""",4.0,null,1.0,13417.0,598.0,24801.0
"""max""",15.0,"""zerglingmovementspeed""",127.0,522030.0,23304.0,36393.0


In [14]:
upgrades_df.select(pl.col("name").value_counts(sort=True)).collect()
# player_id is 0..15
#             0 is only 20 occurrences and it's from campaign
#             From 1 it's actual players.
# name: the name of the upgrade, many are like SprayZerg/Protoss or RewardDance<Unit>
# I checked both  the s2protocol python module and debugged this one and they both are missing most of the upgardes...
# Maybe after a specific version it comes in different event type?

name
struct[2]
"{""SprayZerg"",399567}"
"{""SprayTerran"",240020}"
"{""SprayProtoss"",224421}"
"{""RewardDanceMule"",47355}"
"{""RewardDanceOverlord"",47272}"
"{""zerglingmovementspeed"",45431}"
"{""RewardDanceRoach"",45366}"
"{""RewardDanceViking"",44418}"
"{""RewardDanceStalker"",43733}"


In [15]:
# Clean up a little bit the upgrades:
upgrades_df = upgrades_df.filter(~s2.is_cosmetic_upgrade())
upgrades_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id)).collect()

player_id,name,count,ext_replay_loop,ext_replay_seconds,ext_fs_id
u8,str,i32,i64,u32,u64
4,"""WarpGateResearch""",1,5324,237,16594
2,"""zerglingmovementspeed""",1,5603,250,16594
1,"""WarpGateResearch""",1,6058,270,16594
2,"""ZergMissileWeaponsLevel1""",1,9015,402,16594
2,"""GlialReconstitution""",1,9467,422,16594
3,"""zerglingmovementspeed""",1,9734,434,16594
2,"""EvolveGroovedSpines""",1,10057,448,16594
2,"""ZergGroundArmorsLevel1""",1,10243,457,16594
1,"""ExtendedThermalLance""",1,10621,474,16594


In [16]:
upgrades_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id)).join(
    details_lobby_df,
    on=["player_id", "ext_fs_id"],
).collect()

player_id,name,count,ext_replay_loop,ext_replay_seconds,ext_fs_id,player_name,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_observe,player_result,player_working_set_slot_id,player_hero,title,map_info_sha256,is_blizzard_map,time_utc,time_local_offset,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap,player_scaled_rating,lobby_random_seed,cache_handle_region,cache_handle_extension,cache_handles,ext_fs_sha256,ext_fs_file_name,ext_datetime,ext_protocol_version,user_id
u8,str,i32,i64,u32,u64,str,u8,u32,u32,u64,str,u8,u8,u8,u8,u8,u8,u8,str,u8,str,str,str,bool,i64,i64,i64,u8,u8,u8,i64,i64,u32,i32,u32,str,str,str,str,str,datetime[ns],u32,i64
1,"""WarpGateResearch""",1,6058,270,16594,"""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,0
2,"""zerglingmovementspeed""",1,5603,250,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
2,"""ZergMissileWeaponsLevel1""",1,9015,402,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
2,"""GlialReconstitution""",1,9467,422,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
1,"""ExtendedThermalLance""",1,10621,474,16594,"""Hallo""",2,21298,1,8351209,"""Protoss""",255,180

In [17]:
stats_df.collect_schema()

Schema([('player_id', UInt8),
        ('minerals_current', Int32),
        ('vespene_current', Int32),
        ('minerals_collection_rate', Int32),
        ('vespene_collection_rate', Int32),
        ('workers_active_count', Int32),
        ('minerals_used_in_progress_army', Int32),
        ('minerals_used_in_progress_economy', Int32),
        ('minerals_used_in_progress_technology', Int32),
        ('vespene_used_in_progress_army', Int32),
        ('vespene_used_in_progress_economy', Int32),
        ('vespene_used_in_progress_technology', Int32),
        ('minerals_used_current_army', Int32),
        ('minerals_used_current_economy', Int32),
        ('minerals_used_current_technology', Int32),
        ('vespene_used_current_army', Int32),
        ('vespene_used_current_economy', Int32),
        ('vespene_used_current_technology', Int32),
        ('minerals_lost_army', Int32),
        ('minerals_lost_economy', Int32),
        ('minerals_lost_technology', Int32),
        ('vespene_lost_

In [18]:
stats_df.describe()

statistic,player_id,minerals_current,vespene_current,minerals_collection_rate,vespene_collection_rate,workers_active_count,minerals_used_in_progress_army,minerals_used_in_progress_economy,minerals_used_in_progress_technology,vespene_used_in_progress_army,vespene_used_in_progress_economy,vespene_used_in_progress_technology,minerals_used_current_army,minerals_used_current_economy,minerals_used_current_technology,vespene_used_current_army,vespene_used_current_economy,vespene_used_current_technology,minerals_lost_army,minerals_lost_economy,minerals_lost_technology,vespene_lost_army,vespene_lost_economy,vespene_lost_technology,minerals_killed_army,minerals_killed_economy,minerals_killed_technology,vespene_killed_army,vespene_killed_economy,vespene_killed_technology,food_used,food_made,minerals_used_active_forces,vespene_used_active_forces,minerals_friendly_fire_army,minerals_friendly_fire_economy,minerals_friendly_fire_technology,vespene_friendly_fire_army,vespene_friendly_fire_economy,vespene_friendly_fire_technology,ext_replay_loop,ext_replay_seconds,ext_fs_id
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",2.051812,7866.686273,1871.218374,1491.476389,486.889068,38.178844,259.48911,199.207714,106.928405,129.362639,2.271273,29.628943,2872.44482,4740.50885,2320.472699,1317.025356,115.872364,468.61377,6858.242823,1511.043777,560.392396,3086.801579,38.517382,60.528153,8908.647496,1562.722079,666.981971,4215.069463,29.55089,68.815535,90.869616,127.424473,3032.979461,1312.181916,364.816633,45.907835,24.565703,158.880509,0.447345,5.859324,19419.592424,866.520883,19587.486679
"""std""",1.340163,997127.364669,11317.817672,1087.436226,549.223289,23.320384,420.361616,258.72604,186.670996,260.274758,98.025339,77.428583,94583.823196,5186.753098,9888.372711,2774.990912,3896.808138,1312.340149,82925.522938,4140.79512,2973.611231,8817.891528,1934.669643,792.528261,148821.747462,6212.304476,21206.51969,73352.060321,1548.699072,893.457339,65.748722,228.354541,94581.550375,2770.576608,1351.069457,168.81463,151.449098,754.592224,27.391112,101.544925,27731.087441,1237.992144,10868.414923
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,-12.0,-450.0,0.0,0.0,0.0,-5e7,-371175.0,-69300.0,-4020.0,-294.0,-690.0,-800.0,-1344.0,-1475.0,-900.0,-90.0,-900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-5e7,-1700.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",1.0,133.0,67.0,783.0,67.0,19.0,0.0,0.0,0.0,0.0,0.0,0.0,225.0,2150.0,350.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.0,46.0,250.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5280.0,235.0,10099.0
"""50%""",2.0,365.0,252.0,1343.0,335.0,35.0,105.0,100.0,0.0,0.0,0.0,0.0,1425.0,4350.0,1200.0,450.0,0.0,300.0,500.0,63.0,0.0,50.0,0.0,0.0,500.0,100.0,0.0,75.0,0.0,0.0,78.0,109.0,1500.0,450.0,0.0,0.0,0.0,0.0,0.0,0.0,11040.0,492.0,19797.0
"""75%""",2.0,1400.0,871.0,2155.0,694.0,58.0,375.0,350.0,150.0,175.0,0.0,0.0,3950.0,6800.0,2700.0,1825.0,0.0,750.0,5487.0,1100.0,125.0,1900.0,0.0,0.0,4645.0,950.0,150.0,1625.0,0.0,0.0,148.0,196.0,4100.0,1800.0,50.0,0.0,0.0,0.0,0.0,0.0,20000.0,892.0,29742.0
"""max""",15.0,2.68435651e8,1.323338e6,159028.0,72794.0,200.0,90600.0,90000.0,35950.0,45000.0,90000.0,18500.

In [19]:
stats_df.select(
    pl.col("minerals_used_in_progress_economy").value_counts(sort=True)
).collect()
# player_id From 0..14, mostly 0 and 1, not sure which matches with working_set_slot_id
# minerals_current self explanatory at a given ext_replay_loop for a specific user_id
# vespene_current same as minerals_current
# minerals_collection_rate unsure, seems to be from 0 to 14Ks, 1566 mean. maybe per minute?
# vespene_collection_rate from 0 to 4Ks, 549 mean, as with minerals_collection_rate?
# workers_active_count from 0 to 200 (oddly)
# ------ Following fields are "resource" type:
# ------ - Minerals
# ------ - Vespene
# ------ And for each resource, one stored field per "bucket"
# ------ - Army
# ------ - Economy
# ------ - Technology
# <resource>_used_in_progress_<bucket>
# <resource>_used_current_<bucket>
# <resource>_lost_<bucket>
# <resource>_killed_<bucket>
# <resource>_friendly_fire_<bucket>
# ------ And then two more:
# <resource>_used_active_forces

minerals_used_in_progress_economy
struct[2]
"{0,4055514}"
"{100,974825}"
"{50,912679}"
"{150,776166}"
"{400,661496}"
"{200,494439}"
"{350,453130}"
"{450,448020}"
"{500,379420}"


In [20]:
stats_df.filter(
    [pl.col("ext_fs_id").eq(chosen_fs_id) & pl.col("ext_replay_loop").eq(1)]
).join(
    details_lobby_df,
    on=["player_id", "ext_fs_id"],
).collect()

player_id,minerals_current,vespene_current,minerals_collection_rate,vespene_collection_rate,workers_active_count,minerals_used_in_progress_army,minerals_used_in_progress_economy,minerals_used_in_progress_technology,vespene_used_in_progress_army,vespene_used_in_progress_economy,vespene_used_in_progress_technology,minerals_used_current_army,minerals_used_current_economy,minerals_used_current_technology,vespene_used_current_army,vespene_used_current_economy,vespene_used_current_technology,minerals_lost_army,minerals_lost_economy,minerals_lost_technology,vespene_lost_army,vespene_lost_economy,vespene_lost_technology,minerals_killed_army,minerals_killed_economy,minerals_killed_technology,vespene_killed_army,vespene_killed_economy,vespene_killed_technology,food_used,food_made,minerals_used_active_forces,vespene_used_active_forces,minerals_friendly_fire_army,minerals_friendly_fire_economy,minerals_friendly_fire_technology,…,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_observe,player_result,player_working_set_slot_id,player_hero,title,map_info_sha256,is_blizzard_map,time_utc,time_local_offset,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap,player_scaled_rating,lobby_random_seed,cache_handle_region,cache_handle_extension,cache_handles,ext_fs_sha256,ext_fs_file_name,ext_datetime,ext_protocol_version,user_id
u8,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,u8,u32,u32,u64,str,u8,u8,u8,u8,u8,u8,u8,str,u8,str,str,str,bool,i64,i64,i64,u8,u8,u8,i64,i64,u32,i32,u32,str,str,str,str,str,datetime[ns],u32,i64
1,50,0,0,0,12,0,0,0,0,0,0,0,1000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,12,15,0,0,0,0,0,…,2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,0
2,50,0,0,0,12,0,0,0,0,0,0,0,1050,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,12,14,0,0,0,0,0,…,2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1


In [21]:
unit_born_df.collect_schema()

Schema([('unit_tag_index', UInt32),
        ('unit_tag_recycle', UInt32),
        ('unit_type_name', String),
        ('control_player_id', UInt8),
        ('upkeep_player_id', UInt8),
        ('x', Float32),
        ('y', Float32),
        ('creator_unit_tag_index', UInt32),
        ('creator_unit_tag_recycle', UInt32),
        ('creator_unit_type_name', String),
        ('creator_ability_name', String),
        ('ext_replay_loop', Int64),
        ('ext_replay_seconds', UInt32),
        ('ext_fs_id', UInt64),
        ('player_name', String)])

In [22]:
unit_born_df.describe()

statistic,unit_tag_index,unit_tag_recycle,unit_type_name,control_player_id,upkeep_player_id,x,y,creator_unit_tag_index,creator_unit_tag_recycle,creator_unit_type_name,creator_ability_name,ext_replay_loop,ext_replay_seconds,ext_fs_id,player_name
str,f64,f64,str,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,str
"""count""",1.3422413e8,1.3422413e8,"""134224130""",1.34224045e8,1.34224045e8,1.3422413e8,1.3422413e8,9.2661017e7,9.2661017e7,"""113798674""","""92661045""",1.3422413e8,1.3422413e8,1.3422413e8,"""115350154"""
"""null_count""",0.0,0.0,"""0""",85.0,85.0,0.0,0.0,4.1563113e7,4.1563113e7,"""20425456""","""41563085""",0.0,0.0,0.0,"""18873976"""
"""mean""",502.605515,50.097165,null,1.489138,1.489138,95.351128,92.609993,516.151697,28.31829,null,null,33459.201368,1493.252709,22936.774101,null
"""std""",287.497489,106.911159,null,1.120682,1.120682,43.679554,43.335838,268.829838,59.460307,null,null,36902.548373,1647.404542,10312.480181,null
"""min""",0.0,1.0,"""105mmArtillery2""",0.0,0.0,0.0,0.0,1.0,1.0,"""ACPirateCapitalShip""","""""",0.0,0.0,0.0,"""&lt;0001&gt;<sp/>VOODOO"""
"""25%""",295.0,2.0,null,1.0,1.0,58.0,58.0,306.0,2.0,null,null,11078.0,494.0,14723.0,null
"""50%""",475.0,14.0,null,1.0,1.0,96.0,94.0,507.0,5.0,null,null,19252.0,859.0,26350.0,null
"""75%""",707.0,53.0,null,2.0,2.0,128.0,125.0,718.0,26.0,null,null,43682.0,1950.0,30974.0,null
"""max""",3741.0,5169.0,"""zealot_A22""",15.0,15.0,255.0,255.0,3740.0,4487.0,"""uzig""","""toWarPig""",524287.0,23405.0,36393.0,"""호에엥"""


In [23]:
unit_born_df.select(pl.col("creator_unit_type_name").value_counts(sort=True)).collect()
# unit_tag_index and unit_tag_recycle = used to calculate the unique id of a unit
# unit_type_name = name of unit drone, probe, mineralfield, etc.
# control_player_id Either null or 0..15
#                   When null (almost no instances in this data set) could be the campaign, stuff like K5Kerrigan
#                   When 0 it's the system creating mineral fields, geysers, etc.
#                   From 1..15 is the actual users playing.
# upkeep_player_id  Either null or 0..15
#                   As above, when 0, it's the system creating mineral fields, geysers, etc.
#                   When 1..15 actual users playing
# creator_ability_name Larva, Cocoons, OrbitalCommands, Barracks, Nexus, Lair, creating units.

creator_unit_type_name
struct[2]
"{""LurkerMPBurrowed"",55304068}"
"{null,20425456}"
"{""Egg"",16952694}"
"{""Larva"",7559491}"
"{""Hatchery"",6603726}"
"{""BroodLord"",4179962}"
"{""Barracks"",2310786}"
"{""Nexus"",1633794}"
"{""OrbitalCommand"",1560689}"


In [24]:
unit_born_df.filter(
    [
        pl.col("ext_fs_id").eq(chosen_fs_id)
        & ~pl.col("unit_type_name").str.contains("Beacon")
    ]
).join(
    details_lobby_df.rename({"player_id": "control_player_id"}),
    on=["control_player_id", "ext_fs_id"],
).collect()

unit_tag_index,unit_tag_recycle,unit_type_name,control_player_id,upkeep_player_id,x,y,creator_unit_tag_index,creator_unit_tag_recycle,creator_unit_type_name,creator_ability_name,ext_replay_loop,ext_replay_seconds,ext_fs_id,player_name,player_name_right,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_observe,player_result,player_working_set_slot_id,player_hero,title,map_info_sha256,is_blizzard_map,time_utc,time_local_offset,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap,player_scaled_rating,lobby_random_seed,cache_handle_region,cache_handle_extension,cache_handles,ext_fs_sha256,ext_fs_file_name,ext_datetime,ext_protocol_version,user_id
u32,u32,str,u8,u8,f32,f32,u32,u32,str,str,i64,u32,u64,str,str,u8,u32,u32,u64,str,u8,u8,u8,u8,u8,u8,u8,str,u8,str,str,str,bool,i64,i64,i64,u8,u8,u8,i64,i64,u32,i32,u32,str,str,str,str,str,datetime[ns],u32,i64
320,1,"""Nexus""",1,1,120.0,43.0,null,null,null,null,0,0,16594,"""Hallo""","""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,0
321,1,"""Probe""",1,1,117.0,40.0,null,null,null,null,0,0,16594,"""Hallo""","""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,0
322,1,"""Probe""",1,1,117.0,41.0,null,null,null,null,0,0,16594,"""Hallo""","""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,0
323,1,"""Probe""",1,1,118.0,40.0,null,null,null,null,0,0,16594,"""Hallo""","""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc

In [25]:
(
    unit_born_df.filter(
        [
            pl.col("ext_fs_id").eq(chosen_fs_id)
            & ~pl.col("unit_type_name").str.contains("Beacon")
        ]
    ).join(
        details_lobby_df.rename({"player_id": "control_player_id"}),
        on=["control_player_id", "ext_fs_id"],
    )
    .group_by(pl.col("unit_type_name"), pl.col("player_name"))
        .agg(pl.len().alias("freq"))
        .sort(pl.col("freq"), descending=True)
    .collect()
)

unit_type_name,player_name,freq
str,str,u32
"""InvisibleTargetDummy""","""Sazed""",5311
"""Larva""","""Sazed""",340
"""Egg""","""Sazed""",156
"""Drone""","""Sazed""",88
"""LurkerMPBurrowed""","""Sazed""",70
"""LurkerMP""","""Sazed""",70
"""Probe""","""Hallo""",67
"""Zealot""","""Hallo""",54
"""CreepTumorBurrowed""","""Sazed""",47


In [26]:
# System created units, mineral fields, etc.
unit_born_df.filter(
    [
        pl.col("ext_fs_id").eq(chosen_fs_id)
        & pl.col("ext_replay_loop").eq(0)
        & pl.col("control_player_id").eq(0)
    ]
).select(pl.col("unit_type_name").value_counts(sort=True)).collect()

unit_type_name
struct[2]
"{""MineralField750"",88}"
"{""MineralField"",88}"
"{""VespeneGeyser"",48}"
"{""InhibitorZoneMedium"",8}"
"{""RichMineralField750"",6}"
"{""InhibitorZoneLarge"",6}"
"{""RichMineralField"",6}"
"{""UnbuildableRocksDestructible"",4}"
"{""DestructibleRockEx16x6"",4}"


In [27]:
unit_died_df.collect_schema()

Schema([('unit_died_name', String),
        ('unit_tag_index', UInt32),
        ('unit_tag_recycle', UInt32),
        ('killer_player_id', UInt8),
        ('x', UInt8),
        ('y', UInt8),
        ('unit_killer_name', String),
        ('killer_unit_tag_index', UInt32),
        ('killer_unit_tag_recycle', UInt32),
        ('ext_replay_loop', Int64),
        ('ext_replay_seconds', UInt32),
        ('ext_fs_id', UInt64)])

In [28]:
unit_died_df.describe()

statistic,unit_died_name,unit_tag_index,unit_tag_recycle,killer_player_id,x,y,unit_killer_name,killer_unit_tag_index,killer_unit_tag_recycle,ext_replay_loop,ext_replay_seconds,ext_fs_id
str,str,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64
"""count""","""93023833""",9.3023833e7,9.3023833e7,8.3381454e7,9.3023833e7,9.3023833e7,"""93023833""",7.976215e7,7.976215e7,9.3023833e7,9.3023833e7,9.3023833e7
"""null_count""","""0""",0.0,0.0,9.642379e6,0.0,0.0,"""0""",1.3261683e7,1.3261683e7,0.0,0.0,0.0
"""mean""",null,542.916087,67.102732,1.381273,94.978016,93.454049,null,553.25201,33.59283,42866.768209,1913.199672,24556.498607
"""std""",null,291.015353,122.774063,0.889421,40.46878,38.575225,null,271.315649,72.111808,38724.225315,1728.759925,9745.4368
"""min""","""105mmArtillery2""",0.0,1.0,0.0,0.0,0.0,"""""",0.0,1.0,0.0,0.0,0.0
"""25%""",null,333.0,7.0,1.0,61.0,63.0,null,364.0,3.0,16878.0,753.0,16860.0
"""50%""",null,523.0,27.0,1.0,97.0,95.0,null,569.0,9.0,25099.0,1120.0,28382.0
"""75%""",null,764.0,80.0,2.0,125.0,121.0,null,746.0,34.0,61818.0,2759.0,31327.0
"""max""","""zealot_A22""",3741.0,5169.0,15.0,255.0,255.0,"""zealot_A22""",3714.0,4487.0,524281.0,23405.0,36393.0


In [29]:
unit_died_df.select(pl.col("unit_killer_name").value_counts(sort=True)).collect()
# unit_died_name name of units dying.
#                Most unit names are InvisibleTargetDummy.
#                It seems when lurker or units A-move, a temporary target_dummy is created at that location and when the unit "attacks" it.
#                Q Are tank shots on InvisibleTargetDummy? There's splash radius/damage somewhere maybe?
# unit_killer_name the killer unit if found.

unit_killer_name
struct[2]
"{""LurkerMPBurrowed"",55869272}"
"{"""",19252497}"
"{""Marine"",1676818}"
"{""SiegeTankSieged"",1163809}"
"{""Zergling"",1124629}"
"{""Stalker"",1039970}"
"{""Zealot"",747948}"
"{""Roach"",724336}"
"{""Hydralisk"",718991}"


In [30]:
unit_died_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id)).join(
    details_lobby_df.rename({"player_id": "killer_player_id"}),
    on=["killer_player_id", "ext_fs_id"],
).tail(100).collect()

unit_died_name,unit_tag_index,unit_tag_recycle,killer_player_id,x,y,unit_killer_name,killer_unit_tag_index,killer_unit_tag_recycle,ext_replay_loop,ext_replay_seconds,ext_fs_id,player_name,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_observe,player_result,player_working_set_slot_id,player_hero,title,map_info_sha256,is_blizzard_map,time_utc,time_local_offset,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap,player_scaled_rating,lobby_random_seed,cache_handle_region,cache_handle_extension,cache_handles,ext_fs_sha256,ext_fs_file_name,ext_datetime,ext_protocol_version,user_id
str,u32,u32,u8,u8,u8,str,u32,u32,i64,u32,u64,str,u8,u32,u32,u64,str,u8,u8,u8,u8,u8,u8,u8,str,u8,str,str,str,bool,i64,i64,i64,u8,u8,u8,i64,i64,u32,i32,u32,str,str,str,str,str,datetime[ns],u32,i64
"""InvisibleTargetDummy""",815,16,2,56,160,"""LurkerMPBurrowed""",679,2,17163,766,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
"""InvisibleTargetDummy""",24,46,2,46,157,"""LurkerMPBurrowed""",723,7,17163,766,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
"""InvisibleTargetDummy""",943,9,2,49,162,"""LurkerMPBurrowed""",489,10,17163,766,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
"""InvisibleTargetDummy""",820,44,2,55,164,"""LurkerMPBurrowed""",712,2,17163,766,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab

In [31]:
unit_died_df.select(pl.len()).collect().item()

93023833

In [32]:
(
    unit_died_df
        .filter(pl.col("ext_fs_id").eq(chosen_fs_id))
        .join(
            details_lobby_df.rename({"player_id": "killer_player_id"}),
            on=["killer_player_id", "ext_fs_id"],
        )
        .group_by(pl.col("player_name"), pl.col("unit_killer_name"), pl.col("unit_died_name"))
        .agg(pl.len().alias("freq"))
        .sort(pl.col("freq"), descending=True)
        .collect()
)

player_name,unit_killer_name,unit_died_name,freq
str,str,str,u32
"""Sazed""","""LurkerMPBurrowed""","""InvisibleTargetDummy""",5311
"""Hallo""","""Zealot""","""Probe""",52
"""Sazed""","""LurkerMPBurrowed""","""Broodling""",47
"""Sazed""","""LurkerMPBurrowed""","""Drone""",43
"""Sazed""","""LurkerMPBurrowed""","""Zergling""",23
"""Hallo""","""Colossus""","""Drone""",19
"""Hallo""","""Colossus""","""Broodling""",14
"""Hallo""","""Colossus""","""Zergling""",12
"""Sazed""","""LurkerMPBurrowed""","""Larva""",9


In [33]:
cmd_target_point_df.collect_schema()

Schema([('user_id', Int64),
        ('player_name', String),
        ('cmd_flags', Int64),
        ('abil_link', UInt16),
        ('abil_cmd_index', Int64),
        ('ability', String),
        ('target_point_x', Int64),
        ('target_point_y', Int64),
        ('target_point_z', Int32),
        ('sequence', Int64),
        ('unit_group', UInt32),
        ('unit_name', String),
        ('unit_names', String),
        ('selection_size', UInt32),
        ('ext_replay_loop', Int64),
        ('ext_replay_seconds', UInt32),
        ('ext_fs_id', UInt64)])

In [34]:
cmd_target_point_df.describe()

statistic,user_id,player_name,cmd_flags,abil_link,abil_cmd_index,ability,target_point_x,target_point_y,target_point_z,sequence,unit_group,unit_name,unit_names,selection_size,ext_replay_loop,ext_replay_seconds,ext_fs_id
str,f64,str,f64,f64,f64,str,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64
"""count""",2.2329585e7,"""22329585""",2.2329585e7,1.0562924e7,1.0562924e7,"""22329585""",2.2329585e7,2.2329585e7,2.2329585e7,2.2329585e7,20637.0,"""22329585""","""22329585""",2.2329585e7,2.2329585e7,2.2329585e7,2.2329585e7
"""null_count""",0.0,"""0""",0.0,1.1766661e7,1.1766661e7,"""0""",0.0,0.0,0.0,0.0,2.2308948e7,"""0""","""0""",0.0,0.0,0.0,0.0
"""mean""",2.159997,null,200381.214898,155.578998,1.068906,null,395837.897074,375065.966868,39072.51896,1369.367292,1.5714e7,null,null,8.466191,13536.47012,603.811153,18221.105876
"""std""",2.167587,null,2.2397e6,382.817016,3.002245,null,164746.11475,161650.052039,8630.663466,1402.960974,1.1320e8,null,null,12.697754,12845.032301,573.43902,10531.254754
"""min""",0.0,"""&lt;0001&gt;<sp/>VOODOO""",256.0,42.0,0.0,"""""",0.0,0.0,-57394.0,1.0,68923.0,"""""","""""",0.0,0.0,0.0,0.0
"""25%""",1.0,null,256.0,46.0,0.0,null,260096.0,241664.0,32744.0,557.0,69930.0,null,null,1.0,6384.0,285.0,9057.0
"""50%""",2.0,null,264.0,46.0,0.0,null,392843.0,367483.0,40918.0,1062.0,70037.0,null,null,3.0,10888.0,486.0,18419.0
"""75%""",3.0,null,264.0,172.0,0.0,null,526336.0,504229.0,40954.0,1786.0,70288.0,null,null,10.0,16728.0,746.0,27331.0
"""max""",15.0,"""호에엥""",5.0331904e7,6089.0,29.0,"""move.ScanMove""",1.048575e6,1.048575e6,149253.0,151722.0,3.5588e9,"""ogru""","""ogru""",907.0,522124.0,23309.0,36393.0


In [35]:
cmd_target_point_df.select(pl.col("unit_group").value_counts(sort=True)).collect()
# map_elements(lambda x: f'{x:>064b}', return_dtype=pl.String). # Transform i64 to bits.
# cmd_flags is some bit map, not sure about its meaning. Mostly 0s, a few 1s
# unst_abil is a currently unstable feature of s2protocol-rs that tries to generate the ability strings.
# abil_link: mostly null, then followed by 46, 172, 185, 267
# abil_cmd_index: From 0 to 15, mostly nulls, followed by  0, 1,3,15,4,...
# target_point_x, target_point_y and target_point_z are self explanatory, important that the Z is i32 and x,y are i64 at the source.
# sequence: Unknown 12K values, top are: 5,6,4,7,721,681,...
# unit_group: 20 values, most are null, then 70011 with 44 instances, 69957, 70013, but very very few non nulls.

unit_group
struct[2]
"{null,22308948}"
"{69918,1287}"
"{70037,947}"
"{69919,932}"
"{70038,927}"
"{70048,859}"
"{70049,781}"
"{69902,706}"
"{69903,613}"


In [36]:
cmd_target_point_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id)).join(
    details_lobby_df,
    on=["user_id", "ext_fs_id"],
).sort(by=[pl.col("ext_replay_loop")]).collect()

user_id,player_name,cmd_flags,abil_link,abil_cmd_index,ability,target_point_x,target_point_y,target_point_z,sequence,unit_group,unit_name,unit_names,selection_size,ext_replay_loop,ext_replay_seconds,ext_fs_id,player_name_right,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_observe,player_result,player_working_set_slot_id,player_hero,title,map_info_sha256,is_blizzard_map,time_utc,time_local_offset,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap,player_scaled_rating,lobby_random_seed,cache_handle_region,cache_handle_extension,cache_handles,ext_fs_sha256,ext_fs_file_name,ext_datetime,ext_protocol_version,player_id
i64,str,i64,u16,i64,str,i64,i64,i32,i64,u32,str,str,u32,i64,u32,u64,str,u8,u32,u32,u64,str,u8,u8,u8,u8,u8,u8,u8,str,u8,str,str,str,bool,i64,i64,i64,u8,u8,u8,i64,i64,u32,i32,u32,str,str,str,str,str,datetime[ns],u32,u8
0,"""Hallo""",264,null,null,"""""",469968,175520,49132,1,null,"""Probe""","""Probe""",4,22,0,16594,"""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
0,"""Hallo""",264,null,null,"""""",490729,157809,49132,4,null,"""Nexus""","""Nexus""",1,51,2,16594,"""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
1,"""Sazed""",65800,null,null,"""""",283989,688128,49130,5,null,"""Overlord""","""Overlord""",1,76,3,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,2
1,"""Sazed""",65802,null,null,"""""",273066,805546,32716,6,null,"""Overlord""","""Overlord""",1,82,3,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a7

In [37]:
cmd_target_point_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id)).join(
    details_lobby_df.filter(pl.col("player_race").eq("Zerg"))
    .select(pl.col(["user_id", "ext_fs_id"])),
    on=["user_id", "ext_fs_id"],
).select(
    pl.col("abil_link").value_counts(sort=True)
).collect()

abil_link
struct[2]
"{null,95}"
"{185,21}"
"{46,19}"
"{267,13}"
"{262,4}"
"{43,4}"


In [38]:
cmd_target_point_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id)).join(
    details_lobby_df.filter(pl.col("player_race").eq("Protoss"))
    .select([pl.col("user_id"), pl.col("ext_fs_id")]),
    on=["user_id", "ext_fs_id"],
).select(
    pl.col("abil_link").value_counts(sort=True)
).collect()

abil_link
struct[2]
"{null,89}"
"{172,38}"
"{46,32}"
"{216,18}"
"{234,1}"


In [39]:
cmd_target_unit_df.collect_schema()

Schema([('user_id', Int64),
        ('player_name', String),
        ('cmd_flags', Int64),
        ('abil_link', UInt16),
        ('abil_cmd_index', Int64),
        ('ability', String),
        ('target_unit_flags', UInt16),
        ('tag', UInt32),
        ('snapshot_unit_link', UInt16),
        ('snapshot_control_player_id', Int64),
        ('snapshot_upkeep_player_id', Int64),
        ('snapshot_point_x', Int64),
        ('snapshot_point_y', Int64),
        ('snapshot_point_z', Int32),
        ('sequence', Int64),
        ('unit_name', String),
        ('unit_names', String),
        ('selection_size', UInt32),
        ('target_unit_name', String),
        ('ext_replay_loop', Int64),
        ('ext_replay_seconds', UInt32),
        ('ext_fs_id', UInt64)])

In [40]:
cmd_target_unit_df.describe()

statistic,user_id,player_name,cmd_flags,abil_link,abil_cmd_index,ability,target_unit_flags,tag,snapshot_unit_link,snapshot_control_player_id,snapshot_upkeep_player_id,snapshot_point_x,snapshot_point_y,snapshot_point_z,sequence,unit_name,unit_names,selection_size,target_unit_name,ext_replay_loop,ext_replay_seconds,ext_fs_id
str,f64,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,str,f64,f64,f64
"""count""",7.064893e6,"""7064893""",7.064893e6,3.470117e6,3.470117e6,"""7064893""",7.064893e6,7.064893e6,7.064893e6,7.064893e6,7.064893e6,7.064893e6,7.064893e6,7.064893e6,7.064893e6,"""7064893""","""7064893""",7.064893e6,"""7064893""",7.064893e6,7.064893e6,7.064893e6
"""null_count""",0.0,"""0""",0.0,3.594776e6,3.594776e6,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""",0.0,"""0""",0.0,0.0,0.0
"""mean""",1.996763,null,247665.507128,190.228827,0.28478,null,113.14949,6.3046e7,385.355815,0.961772,0.961885,392956.395089,373531.655616,43401.753023,1239.513141,null,null,4.412756,null,12031.978198,536.645641,18607.801059
"""std""",2.123298,null,2.8691e6,252.165223,0.72012,null,11.530269,5.1356e7,451.972919,1.064516,1.06466,192504.453462,189409.619286,8916.290268,2685.038513,null,null,7.261576,null,14353.888895,640.799282,10576.65233
"""min""",0.0,"""&lt;0001&gt;<sp/>VOODOO""",256.0,42.0,0.0,"""""",111.0,0.0,23.0,0.0,0.0,8192.0,8192.0,-1376.0,1.0,"""""","""""",0.0,"""""",0.0,0.0,0.0
"""25%""",0.0,null,256.0,46.0,0.0,null,111.0,2.3855105e7,100.0,0.0,0.0,215040.0,202752.0,40896.0,358.0,null,null,1.0,null,4634.0,206.0,9583.0
"""50%""",1.0,null,264.0,113.0,0.0,null,111.0,5.6623106e7,134.0,1.0,1.0,391168.0,364544.0,40944.0,824.0,null,null,2.0,null,8715.0,389.0,18810.0
"""75%""",3.0,null,264.0,185.0,0.0,null,111.0,8.7031811e7,547.0,2.0,2.0,565248.0,546816.0,49120.0,1520.0,null,null,4.0,null,14355.0,640.0,27728.0
"""max""",15.0,"""호동이이쁘지""",3.3554698e7,6093.0,23.0,"""move.ScanMove""",175.0,9.77272845e8,8899.0,15.0,15.0,1.040383e6,1.040383e6,162816.0,154897.0,"""now3""","""now3""",464.0,"""Zergling""",505634.0,22572.0,36393.0


In [41]:
cmd_target_unit_df.select(pl.col("ability").value_counts(sort=True)).collect()
# User-ID is 0-indexed
# cmd_flags 8 values, seems bitmap
# abil_link: mostly null, then followed by 46, 172, 185, 267
# abil_cmd_index: 7 possible values, mostly nulls, followed by  0,2,1,3,23,18
# target_unit_flags can be either 111 or 175 I guess some bitmap
# tag: The tag_index+tag_recycle combo
# snapshot_unit_link 321 possible values, top are 545,109,903,111,547,... Maybe related to abil_link?
# snapshot_control_player_id from 0 to 15
# snapshot_upkeep_player_id 0 to 15
# snapshot_pointx, y, z positions, again z is i32.
# sequence 9K values, starts at 2, 1, 6, 3,66, 82,68 maybe bitmap

ability
struct[2]
"{"""",3625066}"
"{""attack.Execute"",970441}"
"{""SpawnLarva.Execute"",688999}"
"{""ChronoBoostEnergyCost.Execute"",300782}"
"{""SCVHarvest.Gather"",209364}"
"{""Transfusion.Execute"",203188}"
"{""CalldownMULE.Execute"",181844}"
"{""Build.Extractor"",159466}"
"{""Build.Assimilator"",114302}"


In [42]:
cmd_target_unit_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id) & (pl.col("ability") != "")).join(
    details_lobby_df,
    on=["user_id", "ext_fs_id"],
).sort(by=[pl.col("ext_replay_loop")]).collect()

user_id,player_name,cmd_flags,abil_link,abil_cmd_index,ability,target_unit_flags,tag,snapshot_unit_link,snapshot_control_player_id,snapshot_upkeep_player_id,snapshot_point_x,snapshot_point_y,snapshot_point_z,sequence,unit_name,unit_names,selection_size,target_unit_name,ext_replay_loop,ext_replay_seconds,ext_fs_id,player_name_right,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_observe,player_result,player_working_set_slot_id,player_hero,title,map_info_sha256,is_blizzard_map,time_utc,time_local_offset,lobby_slot_user_id,lobby_slot_observe,lobby_slot_map_size_x,lobby_slot_map_size_y,lobby_slot_difficulty,lobby_slot_ai_build,lobby_slot_handicap,player_scaled_rating,lobby_random_seed,cache_handle_region,cache_handle_extension,cache_handles,ext_fs_sha256,ext_fs_file_name,ext_datetime,ext_protocol_version,player_id
i64,str,i64,u16,i64,str,u16,u32,u16,i64,i64,i64,i64,i32,i64,str,str,u32,str,i64,u32,u64,str,u8,u32,u32,u64,str,u8,u8,u8,u8,u8,u8,u8,str,u8,str,str,str,bool,i64,i64,i64,u8,u8,u8,i64,i64,u32,i32,u32,str,str,str,str,str,datetime[ns],u32,u8
0,"""Hallo""",256,722,0,"""ChronoBoostEnergyCost.Execute""",111,83886081,82,1,1,493568,178176,49120,54,"""Nexus""","""Nexus""",1,"""""",813,36,16594,"""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
1,"""Sazed""",256,185,2,"""Build.Extractor""",111,33554433,550,0,0,784384,288768,49120,69,"""Drone""","""Drone""",1,"""""",1396,62,16594,"""Sazed""",2,21298,1,8459957,"""Zerg""",255,0,66,255,2,0,0,"""Win""",1,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,1,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,2
0,"""Hallo""",256,172,2,"""Build.Assimilator""",111,54263809,550,0,0,464896,190464,49120,80,"""Probe""","""Probe""",1,"""""",1913,85,16594,"""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""","""c44f2e03eb2038975159e61397e1d34d2bac82ca13d2ad3f433cfe4bbfc6c2c1""",true,132947831407274006,72000000000,0,0,232,232,1,0,100,-36400,3429363492,"""EU""","""s2ma""","""6de41503baccd05656360b6f027db88169fa1989bb6357b1b215a2547939f5fb,421c8aa0f3619b652d23a2735dfee812ab644228235e7a797edecfe8b67da30e,66093832128453efffbb787c80b7d3eec1ad81bde55c83c930dea79c4e505a04,d92dfc48c484c59154270b924ad7d57484f2ab9a47621c7ab16431bf66c53…","""94c36ac44461e69a3aa9789fa08e8a10f922e960f4bf41dceca78b5171dcdc4d""","""/home/seb/SCReplaysOnNVMe/Multiplayer/Arctic Dream LE (296).SC2Replay""",2022-04-18 17:19:00.727400,87702,1
0,"""Hallo""",256,172,2,"""Build.Assimilator""",111,59506689,550,0,0,509952,149504,49120,91,"""Probe""","""Probe""",1,"""""",2142,95,16594,"""Hallo""",2,21298,1,8351209,"""Protoss""",255,180,20,30,2,0,0,"""Win""",0,"""""","""Arctic Dream LE""

In [43]:
cmd_target_unit_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id) & (pl.col("ability") != "")).join(
    details_lobby_df,
    on=["user_id", "ext_fs_id"],
).group_by(pl.col("ability"), pl.col("player_name")).agg(pl.len().alias("freq")).sort(pl.col("freq"),descending=True).collect()

ability,player_name,freq
str,str,u32
"""SpawnLarva.Execute""","""Sazed""",22
"""ChronoBoostEnergyCost.Execute""","""Hallo""",14
"""Build.Assimilator""","""Hallo""",8
"""Build.Extractor""","""Sazed""",6
"""Transfusion.Execute""","""Sazed""",4
"""ViperConsumeStructure.Execute""","""Sazed""",1
"""attack.Execute""","""Sazed""",1
"""attack.Execute""","""Hallo""",1
